This notebook is a downloaded version of the PGD attack on Audioseal. It was tested on google colab.

It receives a set of audio files, runs it through audioseal's watermark to add a watermark, then runs the PGD attack to try to remove the watermark. 

The attack is constrained by epsilon, which is the maximum allowed perturbation. The attack runs for a certain number of steps, and in each step. After each update, it projects the audio back into the epsilon around the original audio.

Perceptual quality is measured using PESQ, which is a standard metric for audio quality. The goal of the attack is to reduce the PESQ score while also reducing the watermark probability.

To get the desired PESQ score, you can adjust the epsilon and the number of steps. A larger epsilon allows for more perturbation, which can lead to a lower PESQ score, but it may also make the audio sound worse. 

ElevenLabs (blackbox api) audios and the attakcs on them can be found here:
https://drive.google.com/drive/folders/1GPzbEmJBIJpZLRxhPp6Et_HhIAtVtHOH?usp=sharing

A collection of output from this code can be found here:
https://drive.google.com/drive/folders/1lO0KQlIb9W7rhZuOKelANmBSQ9NeZFfy?dmr=1&ec=wgc-drive-%5Bmodule%5D-goto



In [ ]:
import os
import shutil
!pip install kaggle

if not os.path.exists('./kaggle'):
    os.makedirs('./kaggle')

if not os.path.exists(os.path.expanduser('~/.kaggle')):
    os.makedirs(os.path.expanduser('~/.kaggle'))

# place a kaggle.json file
shutil.copy('./kaggle/kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

from kaggle.api.kaggle_api_extended import KaggleApi

if not os.path.exists('./kaggle'):
    os.makedirs('./kaggle')

api = KaggleApi()
api.authenticate()

api.dataset_download_files('uwrfkaggler/ravdess-emotional-speech-audio', path='./kaggle', unzip=True)

print("Dataset downloaded and extracted successfully to ./kaggle folder!")


Dataset URL: https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio
Dataset downloaded and extracted successfully to ./kaggle folder!


In [4]:
import numpy as np
import pandas as pd

all_input_files = []
PARENT_FILES_DIR = './kaggle'

# Load .wav audio files from the dataset in ./kaggle folder
for dirname, _, filenames in os.walk(PARENT_FILES_DIR):
    for filename in filenames:
        if filename.endswith(".wav"):
            all_input_files.append(os.path.join(dirname, filename))

print(f"Number of input files: {len(all_input_files)}")

Number of input files: 2880


In [6]:
import sys
!{sys.executable} -m pip install -q torchaudio soundfile matplotlib audioseal

import typing as tp
# import julius
import torch
import torchaudio
import urllib

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

from audioseal import AudioSeal

model = AudioSeal.load_generator("audioseal_wm_16bits")
detector = AudioSeal.load_detector("audioseal_detector_16bits")
model = model.to(device)
detector = detector.to(device)
print("Model and detector loaded")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Model and detector loaded


In [7]:
secret_message = torch.randint(0, 2, (1, 16), dtype=torch.int32)
secret_message = secret_message.to(device)
print(f"Secret message: {secret_message}")

# Function to load an audio file from its file path
def load_audio_file(
    file_path: str
) -> tp.Optional[tp.Tuple[torch.Tensor, int]]:
    try:
        wav, sample_rate = torchaudio.load(file_path)
        return wav, sample_rate
    except Exception as e:
        print(f"Error while loading audio: {e}")
        return None

# Function to generate a watermark for the audio and embed it into a new audio tensor
def generate_watermark_audio(
    tensor: torch.Tensor,
    sample_rate: int
) -> tp.Optional[torch.Tensor]:
    try:
        global model, device, secret_message
        audios = tensor.unsqueeze(0).to(device)
        watermarked_audio = model(audios, sample_rate=sample_rate, message=secret_message.to(device), alpha=1)
        return watermarked_audio


    except Exception as e:
        print(f"Error while watermarking audio: {e}")
        return None

# Function to get the confidence score that an audio tensor was watermarked by Audioseal
def detect_watermark_audio(
    tensor: torch.Tensor,
    sample_rate: int,
    message_threshold: float = 0.50
) -> tp.Optional[float]:
    try:
        global detector, device
        # In our analysis we are not concerned with the hidden/embedded message as of now
        result, _ = detector.detect_watermark(tensor, sample_rate=sample_rate, message_threshold=message_threshold)
        return float(result)
    except Exception as e:
        print(f"Error while detecting watermark: {e}")
        return None

Secret message: tensor([[0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0]], device='cuda:0',
       dtype=torch.int32)


In [8]:
import random
import numpy as np
random.seed(42)
torch.backends.cudnn.benchmark = True
np.random.seed(42)
torch.manual_seed(42)

In [11]:

results = []

for input_file in tqdm(all_input_files):
    try:
        audio, sample_rate = load_audio_file(input_file)

        watermarked_audio = generate_watermark_audio(audio, sample_rate)

        save_path = input_file.replace(".wav", "_watermarked.wav").replace("kaggle", "watermarked_outputs_legit")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        wm = watermarked_audio.squeeze(0).cpu()
        torchaudio.save(save_path, wm, sample_rate)

        reloaded_audio, reloaded_sr = load_audio_file(save_path)
        reloaded_audio = reloaded_audio.unsqueeze(0).to(device)

        confidence_score = detect_watermark_audio(reloaded_audio, reloaded_sr)

        print(f"Confidence score for file {input_file}: {confidence_score}")

        results.append({
            "input_file": input_file,
            "watermarked_file": save_path,
            "sample_rate": sample_rate,
            "confidence_score": float(confidence_score),
            "status": "success"
        })

    except Exception as e:
        print(f"Skipping file {input_file} due to {e}")
        results.append({
            "input_file": input_file,
            "watermarked_file": None,
            "sample_rate": None,
            "confidence_score": None,
            "status": f"error: {e}"
        })

df = pd.DataFrame(results)
df.to_csv("watermark_results.csv", index=False)

print(df)
print(f"\nAverage confidence (successful only): {df['confidence_score'].mean():.4f}")
print(f"Success: {len(df[df.status == 'success'])} / {len(df)}")

  0%|          | 2/2880 [00:00<07:58,  6.02it/s]

Confidence score for file ./kaggle/Actor_18/03-01-03-02-02-02-18.wav: 1.0
Confidence score for file ./kaggle/Actor_18/03-01-05-01-01-01-18.wav: 0.9999999403953552


  0%|          | 4/2880 [00:00<07:19,  6.55it/s]

Confidence score for file ./kaggle/Actor_18/03-01-08-02-02-01-18.wav: 1.0
Confidence score for file ./kaggle/Actor_18/03-01-04-02-01-01-18.wav: 1.0


  0%|          | 6/2880 [00:00<07:00,  6.84it/s]

...

Confidence score for file ./kaggle/Actor_19/03-01-08-01-02-01-19.wav: 1.0
Confidence score for file ./kaggle/Actor_19/03-01-06-01-02-02-19.wav: 1.0


100%|█████████▉| 2879/2880 [11:50<00:00,  6.99it/s]

Confidence score for file ./kaggle/Actor_19/03-01-08-01-01-01-19.wav: 1.0
Confidence score for file ./kaggle/Actor_19/03-01-02-02-01-01-19.wav: 1.0


100%|██████████| 2880/2880 [11:50<00:00,  4.05it/s]


Confidence score for file ./kaggle/Actor_19/03-01-01-01-01-01-19.wav: 1.0
                                      input_file  \
0     ./kaggle/Actor_18/03-01-03-02-02-02-18.wav   
1     ./kaggle/Actor_18/03-01-05-01-01-01-18.wav   
2     ./kaggle/Actor_18/03-01-08-02-02-01-18.wav   
3     ./kaggle/Actor_18/03-01-04-02-01-01-18.wav   
4     ./kaggle/Actor_18/03-01-03-02-01-01-18.wav   
...                                          ...   
2875  ./kaggle/Actor_19/03-01-08-01-02-01-19.wav   
2876  ./kaggle/Actor_19/03-01-06-01-02-02-19.wav   
2877  ./kaggle/Actor_19/03-01-08-01-01-01-19.wav   
2878  ./kaggle/Actor_19/03-01-02-02-01-01-19.wav   
2879  ./kaggle/Actor_19/03-01-01-01-01-01-19.wav   

                                       watermarked_file  sample_rate  \
0     ./watermarked_outputs_legit/Actor_18/03-01-03-...      48000.0   
1     ./watermarked_outputs_legit/Actor_18/03-01-05-...      48000.0   
2     ./watermarked_outputs_legit/Actor_18/03-01-08-...      48000.0   
3     ./water

In [14]:
import glob
from tqdm import tqdm

all_watermarked_files = glob.glob("/content/watermarked_outputs_legit/*/*.wav")
print(len(all_watermarked_files))

1435


In [ ]:
!pip install pesq
from pesq import pesq

def compute_pesq(x_ref, x_adv, sample_rate=16000):
    """
    x_ref, x_adv: [1,1,T] torch tensors
    returns float PESQ score
    """
    ref = x_ref.squeeze().detach().cpu().numpy()
    adv = x_adv.squeeze().detach().cpu().numpy()

    mode = "wb" if sample_rate >= 16000 else "nb"
    return pesq(sample_rate, ref, adv, mode)

import torch.nn.functional as F


def pgd_attack_audioseal(
    x: torch.Tensor,           # [1, 1, T]
    sample_rate: int,
    detector,
    eps: float = 0.002,
    alpha: float = 0.0004,
    steps: int = 10,
):
    # detector.eval()
    detector.train()

    x_adv = x.clone().detach().requires_grad_(True)

    for _ in range(steps):
        result, _ = detector.forward(x_adv, sample_rate=sample_rate)

        watermark_prob = result[:, 1, :]  
        loss = watermark_prob.mean()       

        detector.zero_grad()
        loss.backward()

        with torch.no_grad():
            grad = x_adv.grad
            x_adv -= alpha * grad.sign()

            x_adv = torch.max(
                torch.min(x_adv, x + eps),
                x - eps
            )
            x_adv.clamp_(-1.0, 1.0)

        x_adv.requires_grad_(True)

    return x_adv.detach()

from tqdm import tqdm

# save audio
path_out = "watermarked_pgd_audio_small"
saved_folder = f"/content/{path_out}"
os.makedirs(saved_folder, exist_ok=True)
for input_file in tqdm(all_watermarked_files):
    audio, sample_rate = load_audio_file(input_file)
    audio = audio.unsqueeze(0).to(device)

    base_conf = detect_watermark_audio(audio, sample_rate)

    # Play with epsilon and alpha to get desired PESQ
    audio_adv = pgd_attack_audioseal(
        audio,
        sample_rate,
        detector,
        eps=0.001,
        alpha=0.00001,
        steps=200,
    )

    adv_conf = detect_watermark_audio(audio_adv, sample_rate)
    pesq_score = compute_pesq(audio_adv, audio)
    # save the adv audio
    save_path = input_file.replace(".wav", f"conf_{round(adv_conf,2)}_pesq_{round(pesq_score,2)}_pgd.wav").replace("watermarked_outputs_legit", path_out)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torchaudio.save(save_path, audio_adv.squeeze(0).cpu(), sample_rate)

    print(
        f"{os.path.basename(input_file)} | "
        f"base={base_conf:.4f} → pgd={adv_conf:.4f}, pesq->{pesq_score}"
    )

  0%|          | 1/1435 [00:16<6:41:35, 16.80s/it]

03-01-08-01-02-02-18_watermarked.wav | base=1.0000 → pgd=0.0058, pesq->2.5365331172943115


  0%|          | 2/1435 [00:34<6:57:01, 17.46s/it]

03-01-07-01-02-01-18_watermarked.wav | base=1.0000 → pgd=0.1276, pesq->3.545710325241089


  0%|          | 3/1435 [00:53<7:10:40, 18.04s/it]

...

03-01-02-01-01-01-18_watermarked.wav | base=1.0000 → pgd=0.0000, pesq->2.2639944553375244


  1%|▏         | 20/1435 [06:07<7:08:49, 18.18s/it]

03-01-01-01-01-02-18_watermarked.wav | base=1.0000 → pgd=0.0000, pesq->2.0263047218322754


  1%|▏         | 20/1435 [06:17<7:25:06, 18.87s/it]


KeyboardInterrupt: 

In [50]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Source and destination paths
path_to_copy = "/content/watermarked_pgd_audio_small"
dest_path = "/content/drive/MyDrive/adv_outputs_pgd_small"


# Copy entire folder
shutil.copytree(path_to_copy, dest_path, dirs_exist_ok=True)

print("Copy completed:", dest_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copy completed: /content/drive/MyDrive/adv_outputs_pgd_small
